# v0 · The whole document in every prompt

The baseline: send the full Acme 10-Q plus each question to the model, the way a chat app handles an uploaded PDF, and read the `usage` field.

Watch **input tokens stay at the size of the document** on every follow-up. This is v0. Every later version is measured against it.

**Setup:** see `README.md` (`pip install -e .`, then `.env` names the model and server). Run with the *Agentic RAG demos* kernel.
The filing is synthetic. Regenerate it with `python data/make_acme_10q.py`.

In [ ]:
import os
import time
from pathlib import Path

import litellm
import pandas as pd

from tools.settings import LLM_KWARGS, MODEL, price  # reads .env: model, server, price

doc = Path("data/acme_10q.txt").read_text()
doc_tokens = litellm.token_counter(model=MODEL, text=doc)
print(f"Model:    {MODEL}")
print(f"Document: {len(doc):,} characters ≈ {doc_tokens:,} tokens")

## One conversation, three follow-ups

Chat APIs are stateless. Every turn resends the system prompt, **the whole document**, and the conversation so far.
`ask()` streams the answer so we can time the first token, then prints what the provider billed.

In [ ]:
SYSTEM = "You answer questions about the attached quarterly filing. Be brief and cite the section you used."

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": f"<document name=\"Acme 10-Q\">\n{doc}\n</document>"},
    {"role": "assistant", "content": "I've read the filing. What would you like to know?"},
]
runs = []


def ask(question: str) -> None:
    messages.append({"role": "user", "content": question})
    start = time.perf_counter()
    ttft = None
    chunks = []
    print(f"› {question}\n")
    for chunk in litellm.completion(model=MODEL, messages=messages, stream=True,
                                    stream_options={"include_usage": True}, **LLM_KWARGS):
        chunks.append(chunk)
        delta = chunk.choices[0].delta.content if chunk.choices else None
        if delta:
            ttft = ttft or time.perf_counter() - start
            print(delta, end="", flush=True)
    reply = litellm.stream_chunk_builder(chunks, messages=messages)
    messages.append({"role": "assistant", "content": reply.choices[0].message.content})

    usage = reply.usage
    details = getattr(usage, "prompt_tokens_details", None)
    cached = getattr(details, "cached_tokens", 0) or 0
    cost = price(usage.prompt_tokens, usage.completion_tokens)  # list price, or PRICE_AS's for a local model
    runs.append({"question": question, "prompt_tokens": usage.prompt_tokens, "cached_tokens": cached,
                 "completion_tokens": usage.completion_tokens, "ttft_s": round(ttft or 0, 2),
                 "cost_usd": cost})
    print(f"\n\nprompt_tokens={usage.prompt_tokens:,}  cached={cached:,}  "
          f"completion_tokens={usage.completion_tokens:,}  ttft={ttft or 0:.2f}s  "
          f"cost={'$%.4f' % cost if cost is not None else 'n/a'}")

In [ ]:
ask("What was Q3 revenue?")

In [ ]:
ask("How about Q2?")

In [ ]:
ask("And the margin?")

## What we paid

In [ ]:
df = pd.DataFrame(runs)
df.loc["total"] = df.sum(numeric_only=True)
df.loc["total", "question"] = ""
df

## Takeaways

- **Input tokens ≈ document size, every turn.** Three short questions cost ~300K input tokens to read ~100 tokens of relevant facts.
- **Follow-ups make it worse.** The history grows on top of the document, so turn 3 is the largest prompt.
- **Prompt caching softens the bill, not the problem.** OpenAI and Anthropic discount a repeated prefix (`cached_tokens`), but the model still reads the whole filing, the context window still fills, and a new document or user starts cold.
- **Long prompts are slower and less accurate.** Time to first token grows with input, and facts in the middle get missed.

Next, v1 sends only the chunks that matter.